# Lane Seven — Demand Forecasting
# **v8 — Allocation Weight Grid Search** 🔬

> **Hypothesis:** The v7.6 12-month allocation window over-indexes on stale mix data.
> A shorter 6-month window with stronger recency weighting may improve March/April WMAPE.

## What v8 changes vs v7.6

| Layer | v7.6 | v8 |
|-------|------|----|
| Upstream model | StyleCode XGB/LGB | **same — reused** |
| Calibration | Global [0.95,1.05] | **same — reused** |
| Allocation window | 12 months, 3/2/1 | **6 months, varies** |
| Allocation variants | 1 | **9 + baseline = 10** |

## Important: apples-to-apples design

- StyleCode forecasts are generated **once** per horizon (same as v7.6)
- Calibration is applied **once** (same as v7.6)
- **Only allocation parameters vary** across the grid
- All variants scored against the same holdout population

## Baseline

```
baseline_v76_L12_3_2_1  (lookback=12, w_recent=3, w_mid=2, w_old=1)
```

## Grid (L=6 variants)

```
v8_L6_3_2_1   v8_L6_4_2_1   v8_L6_5_2_1   v8_L6_6_2_1
v8_L6_5_3_1   v8_L6_6_3_1   v8_L6_4_3_1
v8_L6_5_1_1   v8_L6_6_1_1
```

## Section 0 — Environment setup

In [ ]:
import subprocess, sys
packages = ['pandas','numpy','scikit-learn','xgboost','lightgbm','statsmodels','openpyxl']
subprocess.check_call([sys.executable,'-m','pip','install','--quiet']+packages)
print('All packages installed.')

In [ ]:
import sys, logging
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path('.').resolve()
DATA_DIR     = NOTEBOOK_DIR / 'data'
OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

logging.basicConfig(
    level=logging.INFO, format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S', handlers=[logging.StreamHandler(sys.stdout)], force=True,
)
for _noisy in ['lightgbm','xgboost','prophet','cmdstanpy']:
    logging.getLogger(_noisy).setLevel(logging.ERROR)

GOLD_OPTIONS = ['gold_fact_monthly_demand_v2.csv','gold_fact_monthly_demand.csv']
gold_present = [f for f in GOLD_OPTIONS if (DATA_DIR/f).exists()]
print(f'DATA_DIR   : {DATA_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'  {chr(10003) if gold_present else chr(10007)}  {gold_present[0] if gold_present else "MISSING"}')
print(f'  {chr(10003) if (DATA_DIR/"dim_date.csv").exists() else chr(10007)}  dim_date.csv')
print(f'  {chr(10003) if (DATA_DIR/"dim_product.csv").exists() else chr(10007)}  dim_product.csv')
if not gold_present:
    raise FileNotFoundError('Gold demand file missing.')
print('Setup complete.')

## Section 1 — Configuration

In [ ]:
from lane7_forecast.forecast_adjustments import get_config
from lane7_forecast.allocation_grid_search_v8 import WEIGHT_GRID, BASELINE_PARAMS

PHASE = 1

# ── Dynamic holdout detection from dim_date.csv ────────────────────────────────
_dim_date_check = pd.read_csv(DATA_DIR / 'dim_date.csv', parse_dates=['MonthStart'])
_dim_date_check['MonthStart'] = (
    _dim_date_check['MonthStart']
    .dt.to_period('M')
    .dt.to_timestamp()
)

_candidate_split_cols = [
    'Split', 'TrainHoldout', 'DataSplit', 'DatasetSplit', 'Set', 'IsTrain',
]

_holdout_mask = None
_holdout_source_col = None

for _col in _candidate_split_cols:
    if _col not in _dim_date_check.columns:
        continue
    if _col == 'IsTrain':
        _vals = _dim_date_check[_col]
        if _vals.dtype == bool:
            _mask = (_vals == False)
        else:
            _mask = _vals.astype(str).str.strip().str.lower().isin(['0', 'false', 'holdout'])
    else:
        _mask = (
            _dim_date_check[_col]
            .astype(str).str.strip().str.upper()
            .eq('HOLDOUT')
        )
    if _mask.any():
        _holdout_mask = _mask
        _holdout_source_col = _col
        break

if _holdout_mask is None:
    raise ValueError(
        'Could not detect holdout months from dim_date.csv. '
        'Expected Split/TrainHoldout/DataSplit/DatasetSplit/Set containing HOLDOUT, '
        'or IsTrain containing 0/False.'
    )

HOLDOUT_MONTHS = sorted(_dim_date_check.loc[_holdout_mask, 'MonthStart'].unique())
HOLDOUT_MONTHS = [pd.Timestamp(m) for m in HOLDOUT_MONTHS]
if not HOLDOUT_MONTHS:
    raise ValueError('No holdout months detected.')

HOLDOUT_START    = HOLDOUT_MONTHS[0].strftime('%Y-%m-%d')
N_HOLDOUT_MONTHS = len(HOLDOUT_MONTHS)

print(f'Holdout source column : {_holdout_source_col}')
print(f'Detected {N_HOLDOUT_MONTHS} holdout months:')
for m in HOLDOUT_MONTHS: print(f'  {m.strftime("%Y-%m")}')

# v7.6 (and v8) shared config
ADJ_CONFIG = get_config(
    shrink_threshold=1.3, shrink_factor=0.75, recursive_alpha=0.6,
    blend_threshold=0.20, blend_weight=0.7, intermittent_cap_multiplier=2.0,
)

V76_CALIB_CONFIG = dict(
    min_factor=0.95, max_factor=1.05, max_allowed_wmape_regression=0.25
)

print(f'\nWeight grid: {len(WEIGHT_GRID)} L6 variants + 1 baseline = {len(WEIGHT_GRID)+1} total')
print(f'Baseline: lookback={BASELINE_PARAMS["lookback_months"]}  '
      f'w_recent={BASELINE_PARAMS["w_recent"]}  '
      f'w_mid={BASELINE_PARAMS["w_mid"]}  '
      f'w_old={BASELINE_PARAMS["w_old"]}')

## Section 2 — Load Data and Build StyleCode Panel

In [ ]:
from lane7_forecast.pipeline import run_v7_prep, run_cv

# Use BASELINE panel params (lookback=12) for the panel build.
# Allocation shares will be recomputed per variant in Section 5.
scode_prep = run_v7_prep(
    data_dir=DATA_DIR, phase=PHASE,
    lookback_months=BASELINE_PARAMS['lookback_months'],
    min_lookback_months=BASELINE_PARAMS['min_lookback_months'],
)

tables      = scode_prep['tables']
gold_df     = tables['demand']
dim_prod_df = scode_prep['dim_product']
dim_date_df = tables['dim_date']

scode_segments = scode_prep['segments']
scode_segments.to_csv(OUTPUT_DIR/'v8_stylecode_segments.csv', index=False)

print(f'StyleCode panel: {scode_prep["panel_seg"]["SKU"].nunique():,} StyleCodes')
for seg, n in scode_segments['Segment'].value_counts().items():
    print(f'  {seg:<15} {n:5d}')

In [ ]:
USE_EXISTING_CV = False
_best_path = OUTPUT_DIR / 'v8_best_models.csv'
_v76_best  = OUTPUT_DIR / 'v7_6_best_models.csv'
_v74_best  = OUTPUT_DIR / 'v7_4_best_models.csv'

v8_bm = {}

if USE_EXISTING_CV and _best_path.exists():
    print('Loading cached v8 CV results...')
    best_models_v8 = pd.read_csv(_best_path)
    for h in [1,3]:
        v8_bm[h] = best_models_v8[best_models_v8['HorizonMonths']==h].copy()
elif _v76_best.exists():
    print(f'Reusing v7.6 best models from {_v76_best.name}')
    best_models_v8 = pd.read_csv(_v76_best)
    for h in [1,3]: v8_bm[h] = best_models_v8[best_models_v8['HorizonMonths']==h].copy()
elif _v74_best.exists():
    print(f'Reusing v7.4 best models from {_v74_best.name}')
    best_models_v8 = pd.read_csv(_v74_best)
    for h in [1,3]: v8_bm[h] = best_models_v8[best_models_v8['HorizonMonths']==h].copy()
else:
    print('Running StyleCode CV...')
    for h, cfg in {1:dict(n_folds=6,step_months=3,min_train_months=24),
                   3:dict(n_folds=5,step_months=3,min_train_months=24)}.items():
        print(f'--- H={h} ---')
        cv_df, best_df = run_cv(scode_prep, horizon_months=h, **cfg)
        v8_bm[h] = best_df

best_models_v8 = pd.concat([df for df in v8_bm.values() if not df.empty], ignore_index=True)
best_models_v8.to_csv(OUTPUT_DIR/'v8_best_models.csv', index=False)

print('\n=== v8 Best Models ===')
print(best_models_v8[['Segment','HorizonMonths','BestModel','mean_WMAPE']].round(2).to_string(index=False))

## Section 3 — Generate Shared Upstream Forecasts (once)

The same StyleCode forecasts (raw → calibrated) are shared across **all** allocation variants.
Only allocation parameters will differ.

In [ ]:
from lane7_forecast.pipeline import run_forecasts
from lane7_forecast.data_prep import _resolve_training_end, build_panel
from lane7_forecast.segmentation import segment_skus, attach_segment

train_end       = _resolve_training_end(dim_date_df, phase=1)
standalone_skus = scode_prep.get('stylecode_standalone_skus', [])

v8_raw_fc = {}
v8_sa_fc  = {}

for horizon in [3, 1]:
    bm_h = best_models_v8[best_models_v8['HorizonMonths']==horizon]
    if bm_h.empty:
        print(f'H={horizon}: no best model — skipping.')
        continue

    raw_fc = run_forecasts(
        prep=scode_prep, best_models_df=bm_h, horizon_months=horizon,
        forecast_start=HOLDOUT_START, n_forecast_months=N_HOLDOUT_MONTHS,
        phase=PHASE, model_version=f'v8-shared-raw-H{horizon}',
        output_path=None, append=False, adjustment_config=ADJ_CONFIG,
    )
    v8_raw_fc[horizon] = raw_fc
    raw_fc.to_csv(OUTPUT_DIR/f'v8_shared_stylecode_raw_H{horizon}.csv', index=False)
    print(f'H={horizon} raw: {len(raw_fc):,} rows, {raw_fc["Key"].nunique():,} StyleCodes')

    if standalone_skus:
        sa_gold = gold_df[gold_df['SKU'].isin(standalone_skus)].copy()
        if not sa_gold.empty:
            sa_p = build_panel(sa_gold, dim_date_df, phase=PHASE, dim_product_df=dim_prod_df)
            sa_s = segment_skus(sa_p)
            sa_prep = {'tables':tables,'panel':sa_p,'segments':sa_s,
                       'panel_seg':attach_segment(sa_p,sa_s)}
            v8_sa_fc[horizon] = run_forecasts(
                prep=sa_prep, best_models_df=bm_h, horizon_months=horizon,
                forecast_start=HOLDOUT_START, n_forecast_months=N_HOLDOUT_MONTHS,
                phase=PHASE, model_version=f'v8-sa-H{horizon}',
                output_path=None, append=False, adjustment_config=ADJ_CONFIG,
            )

print('\nShared StyleCode forecasts generated. All allocation variants will use these.')

## Section 4 — Apply v7.6 Calibration (shared)

In [ ]:
from lane7_forecast.global_bias_control_v76 import (
    build_global_calibration_table, apply_global_calibration, validate_global_calibration_table,
)

# Load calibration evidence (same as v7.6)
_bt_path   = OUTPUT_DIR / 'v7_5_stylecode_backtest_predictions.csv'
_hold_path = OUTPUT_DIR / 'v7_4_holdout_predictions.csv'

backtest_df = pd.read_csv(_bt_path)   if _bt_path.exists()   else None
holdout_df  = pd.read_csv(_hold_path) if _hold_path.exists() else None

if backtest_df is not None:
    print(f'Calibration evidence: v7.5 backtest ({len(backtest_df):,} rows)')
elif holdout_df is not None:
    print('Calibration evidence: v7.4 holdout (diagnostic fallback)')
else:
    print('No calibration evidence — will use factor=1.0')

actuals_df_full = pd.read_csv(
    next(DATA_DIR/f for f in ['gold_fact_monthly_demand_v2.csv','gold_fact_monthly_demand.csv']
         if (DATA_DIR/f).exists()),
    parse_dates=['MonthStart']
)
actuals_df_full['MonthStart'] = actuals_df_full['MonthStart'].dt.to_period('M').dt.to_timestamp()
actuals_df_full['SKU'] = actuals_df_full['SKU'].astype(str).str.strip()

_gate_fc = v8_raw_fc.get(3, pd.DataFrame())

calib_table = build_global_calibration_table(
    backtest_predictions_df=backtest_df,
    holdout_predictions_df=holdout_df,
    raw_scode_forecasts_df=_gate_fc,
    actuals_df=actuals_df_full,
    dim_product_df=dim_prod_df,
    holdout_months=HOLDOUT_MONTHS,
    horizon_months_list=[1, 3],
    **V76_CALIB_CONFIG,
)
calib_table.to_csv(OUTPUT_DIR/'v8_shared_calibration_table.csv', index=False)

print('=== Shared Calibration Table ===')
print(calib_table[['HorizonMonths','raw_bias_ratio','proposed_factor','final_factor',
                    'calibration_applied','rejection_reason','evidence_source']].to_string(index=False))

# Apply calibration to shared raw forecasts
v8_cal_fc = {}
for horizon in [3, 1]:
    if horizon not in v8_raw_fc: continue
    calib_h = calib_table[calib_table['HorizonMonths']==horizon]
    v8_cal_fc[horizon] = apply_global_calibration(
        scode_forecasts_df=v8_raw_fc[horizon],
        calibration_df=calib_h,
    )
    v8_cal_fc[horizon].to_csv(OUTPUT_DIR/f'v8_shared_stylecode_calibrated_H{horizon}.csv', index=False)
    n_adj = int(v8_cal_fc[horizon]['CalibrationApplied'].sum())
    print(f'H={horizon}: {n_adj} rows calibrated')

print('\nShared calibrated StyleCode forecasts ready.')

## Section 5 — Run Allocation Grid Search

> Each variant reallocates the **same calibrated StyleCode forecasts** using different
> recency weights. Models are NOT retrained. This is a pure allocation experiment.

In [ ]:
from lane7_forecast.allocation_grid_search_v8 import (
    run_allocation_grid, build_grid_results_df, build_top3_df, WEIGHT_GRID,
)

print(f'Running allocation grid: {len(WEIGHT_GRID)+1} variants (1 baseline + {len(WEIGHT_GRID)} L6)')
print()

variant_results = run_allocation_grid(
    calibrated_scode_fc=v8_cal_fc,
    gold_df=gold_df,
    dim_product_df=dim_prod_df,
    train_end=train_end,
    holdout_months=HOLDOUT_MONTHS,
    actuals_df=actuals_df_full,
    weight_grid=WEIGHT_GRID,
    standalone_fc=v8_sa_fc if v8_sa_fc else None,
)

print(f'\nGrid complete: {len(variant_results)} variants ran.')

## Section 6 — Build and Save Grid Results

In [ ]:
grid_df = build_grid_results_df(variant_results, HOLDOUT_MONTHS)

# Define column order for display
month_cols = []
for m in HOLDOUT_MONTHS:
    m_name = m.strftime('%b').capitalize()
    month_cols += [f'H1_{m_name}_WMAPE', f'H3_{m_name}_WMAPE']

display_cols = [
    'Rank_H3','Variant','lookback_months','w_recent','w_mid','w_old',
    'Overall_H3_WMAPE','Overall_H1_WMAPE','BiasRatio','ValidationPassed',
] + [c for c in month_cols if c in grid_df.columns]

grid_df.to_csv(OUTPUT_DIR/'v8_allocation_grid_results.csv', index=False)

print('=== ALLOCATION GRID RESULTS (ranked by Overall_H3_WMAPE) ===')
print(grid_df[display_cols].to_string(index=False))

## Section 7 — Top 3 Variants

In [ ]:
top3_df = build_top3_df(grid_df, HOLDOUT_MONTHS)
top3_df.to_csv(OUTPUT_DIR/'v8_top3_allocation_variants.csv', index=False)

print('=== TOP 3 ALLOCATION VARIANTS ===')
print(top3_df.to_string(index=False))
print()

# Identify winner
best_row = grid_df[grid_df['ValidationPassed'] == True].nsmallest(1, 'Overall_H3_WMAPE').iloc[0] \
    if (grid_df['ValidationPassed'] == True).any() \
    else grid_df.nsmallest(1, 'Overall_H3_WMAPE').iloc[0]

BEST_VARIANT_NAME = best_row['Variant']
BEST_PARAMS = next(
    (r['params'] for r in variant_results if r['variant_name'] == BEST_VARIANT_NAME),
    None
)

# Check if best beats v7.6 baseline
baseline_row = grid_df[grid_df['Variant'].str.startswith('baseline_v76')]
baseline_h3  = float(baseline_row['Overall_H3_WMAPE'].values[0]) if not baseline_row.empty else float('nan')
best_h3      = float(best_row['Overall_H3_WMAPE'])

print(f'Best variant : {BEST_VARIANT_NAME}')
print(f'Best H3 WMAPE: {best_h3:.2f}%')
print(f'Baseline     : {baseline_h3:.2f}%')
if best_h3 < baseline_h3:
    print(f'  ✓ v8 improves over v7.6 baseline by {baseline_h3 - best_h3:.2f}pp')
else:
    print(f'  ✗ v8 does NOT beat v7.6 baseline. Baseline remains champion.')

## Section 8 — Best Variant: Full Production Outputs

In [ ]:
from lane7_forecast.allocation_grid_search_v8 import (
    build_v8_production_sku_table,
    build_v8_validation_report,
    build_variant_error_decomp,
    score_variant_holdout,
)

best_result = next(r for r in variant_results if r['variant_name'] == BEST_VARIANT_NAME)

# ── Production SKU table ──────────────────────────────────────────────────────
best_sku_table = build_v8_production_sku_table(
    sku_fc_dict=best_result['sku_fc'],
    dim_product_df=dim_prod_df,
    variant_name=BEST_VARIANT_NAME,
    params=BEST_PARAMS,
    calibration_df=calib_table,
)
best_sku_table.to_csv(OUTPUT_DIR/'v8_best_production_sku_forecasts.csv', index=False)
print(f'Production SKU table: {len(best_sku_table):,} rows, {best_sku_table["SKU"].nunique():,} SKUs')

# ── Holdout evaluation ────────────────────────────────────────────────────────
best_eval_df = best_result['eval_df']
best_eval_df.to_csv(OUTPUT_DIR/'v8_best_holdout_evaluation.csv', index=False)
print('\n=== Best Holdout Evaluation ===')
print(best_eval_df.to_string(index=False))

# ── Error decomposition ───────────────────────────────────────────────────────
best_decomp_parts = []
ref_scode_fc = next(iter(v8_cal_fc.values()), pd.DataFrame())  # H=3 preferred
for h in [3, 1]:
    if h not in best_result['sku_fc'] or h not in best_result['scol_fc']: continue
    best_decomp_parts.append(build_variant_error_decomp(
        actuals_df=actuals_df_full, dim_product_df=dim_prod_df,
        scode_fc=v8_cal_fc.get(h, pd.DataFrame()),
        scol_fc=best_result['scol_fc'][h],
        sku_fc=best_result['sku_fc'][h],
        holdout_months=HOLDOUT_MONTHS,
    ))
best_decomp = pd.concat(best_decomp_parts, ignore_index=True) if best_decomp_parts else pd.DataFrame()
best_decomp.to_csv(OUTPUT_DIR/'v8_best_error_decomposition.csv', index=False)
print('\n=== Best Error Decomposition ===')
print(best_decomp.sort_values(['HorizonMonths','MonthStart','Level']).to_string(index=False))

## Section 9 — Best Variant Validation Report

In [ ]:
ref_h = 3 if 3 in best_result['sku_fc'] else next(iter(best_result['sku_fc']))

best_val = build_v8_validation_report(
    scode_fc=v8_cal_fc.get(ref_h, pd.DataFrame()),
    scol_fc=best_result['scol_fc'][ref_h],
    sku_fc=best_result['sku_fc'][ref_h],
    calibration_df=calib_table,
    min_factor=V76_CALIB_CONFIG['min_factor'],
    max_factor=V76_CALIB_CONFIG['max_factor'],
)
best_val.to_csv(OUTPUT_DIR/'v8_best_validation_report.csv', index=False)

print('=== Best Variant Validation Report ===')
print(best_val.to_string(index=False))
n_fail = (~best_val['passed']).sum()
print()
print('✅ ALL CHECKS PASSED' if n_fail == 0 else f'❌ {n_fail} check(s) FAILED')

## Section 10 — Version Comparison: v8 vs v7.4 / v7.5 / v7.6

In [ ]:
from lane7_forecast.allocation_grid_search_v8 import build_v8_version_comparison

# Load prior version evaluations (graceful if missing)
def _load_eval(fname):
    p = OUTPUT_DIR / fname
    return pd.read_csv(p) if p.exists() else None

_v74_eval = _load_eval('v7_4_holdout_evaluation.csv')
_v75_eval = _load_eval('v7_5_holdout_evaluation.csv')
_v76_eval = _load_eval('v7_6_holdout_evaluation.csv')

for v, df in [('v7.4', _v74_eval), ('v7.5', _v75_eval), ('v7.6', _v76_eval)]:
    if df is not None:
        print(f'  Loaded {v}: {len(df)} rows')
    else:
        print(f'  {v}: not found — will skip in comparison')

comparison = build_v8_version_comparison(
    best_eval=best_eval_df,
    holdout_months=HOLDOUT_MONTHS,
    best_variant_name=BEST_VARIANT_NAME,
    v74_eval=_v74_eval,
    v75_eval=_v75_eval,
    v76_eval=_v76_eval,
)
comparison.to_csv(OUTPUT_DIR/'v8_vs_prior_versions.csv', index=False)

print()
print('=== v8 vs PRIOR VERSIONS ===')
print(comparison.to_string(index=False))

## Section 11 — Client Readiness Summary

In [ ]:
print('=' * 70)
print('LANE SEVEN DEMAND FORECASTING — v8 ALLOCATION GRID SEARCH SUMMARY')
print('=' * 70)
print(f'Holdout months: {[m.strftime("%Y-%m") for m in HOLDOUT_MONTHS]}')
print(f'Variants tested: {len(variant_results)} ({len(WEIGHT_GRID)} L6 + 1 baseline)')
print(f'Best variant: {BEST_VARIANT_NAME}')
print('=' * 70)
print()

def _get_h3(eval_df, m_str):
    if eval_df is None or eval_df.empty: return float('nan')
    r = eval_df[(eval_df['HorizonMonths']==3) & (eval_df['MonthStart']==m_str)]
    return round(float(r['WMAPE'].iloc[0]),2) if not r.empty else float('nan')

print('=== H3 WMAPE by month — all versions ===')
header = f'{"Version":<30}' + ''.join(f'  {m.strftime("%b"):>8}' for m in HOLDOUT_MONTHS) + '  Overall'
print(header)
print('-' * len(header))

for label, df in [
    ('v7.4', _v74_eval),
    ('v7.5', _v75_eval),
    ('v7.6', _v76_eval),
    (BEST_VARIANT_NAME, best_eval_df),
]:
    vals = [_get_h3(df, m.strftime('%Y-%m')) for m in HOLDOUT_MONTHS]
    overall = round(np.nanmean(vals), 2) if vals else float('nan')
    month_str = ''.join(f'  {v:>7.1f}%' if not np.isnan(v) else '      n/a' for v in vals)
    print(f'{label:<30}{month_str}  {overall:>7.1f}%')

print()
print('=== Verdict ===')
best_h3_all   = float(best_eval_df[best_eval_df['HorizonMonths']==3]['WMAPE'].mean())
v76_h3_all    = float(_v76_eval[_v76_eval['HorizonMonths']==3]['WMAPE'].mean()) if _v76_eval is not None else float('nan')

green, amber, red = [], [], []
if not np.isnan(best_h3_all) and not np.isnan(v76_h3_all):
    delta = best_h3_all - v76_h3_all
    if delta < -0.5:
        green.append(f'v8 beats v7.6 overall H3 by {-delta:.2f}pp ({best_h3_all:.1f}% vs {v76_h3_all:.1f}%)')
    elif delta < 0:
        amber.append(f'v8 marginally better than v7.6 ({delta:+.2f}pp) — within noise')
    else:
        red.append(f'v8 does NOT beat v7.6 baseline (delta={delta:+.2f}pp) — retain v7.6 as champion')

if best_result['validation_passed']:
    green.append('Validation passed for best variant')
else:
    red.append('Validation FAILED for best variant — review before production use')

tot_act  = best_eval_df['TotalActual'].sum()
tot_pred = best_eval_df['TotalPredicted'].sum()
bias_v8  = round(tot_pred/tot_act, 4) if tot_act > 0 else float('nan')
if abs(bias_v8 - 1.0) < 0.05:  green.append(f'Bias near-neutral (ratio={bias_v8:.3f})')
elif abs(bias_v8 - 1.0) < 0.10: amber.append(f'Mild bias (ratio={bias_v8:.3f})')
else:                            red.append(f'Significant bias (ratio={bias_v8:.3f})')

for g in green: print(f'  ✓ {g}')
for a in amber: print(f'  ⚠ {a}')
for r in red:   print(f'  ✗ {r}')
print()
if not red:
    if not amber: print('✅ VERDICT: v8 is READY. Best variant recommended for production.')
    else: print('⚠  VERDICT: v8 shows marginal improvement. Consider retaining v7.6.')
else:
    print('❌ VERDICT: v8 does NOT improve on v7.6. Retain v7.6 as production champion.')

---
## Output Files

| File | Description |
|------|-------------|
| `v8_allocation_grid_results.csv` | All 10 variants ranked by Overall_H3_WMAPE |
| `v8_top3_allocation_variants.csv` | Top 3 variants |
| `v8_best_production_sku_forecasts.csv` | SKU forecasts from best variant |
| `v8_best_holdout_evaluation.csv` | Holdout eval for best variant |
| `v8_best_error_decomposition.csv` | WMAPE at SC/SColor/SKU for best variant |
| `v8_best_validation_report.csv` | Validation checks for best variant |
| `v8_vs_prior_versions.csv` | v8 vs v7.4 / v7.5 / v7.6 |
| `v8_shared_calibration_table.csv` | Shared calibration (same as v7.6) |
| `v8_shared_stylecode_raw_H*.csv` | Pre-calibration StyleCode forecasts |
| `v8_shared_stylecode_calibrated_H*.csv` | Post-calibration StyleCode forecasts |

## Required Prior Comparison Files (place in `outputs/`)

| File | Required? |
|------|-----------|
| `v7_4_holdout_evaluation.csv` | Recommended |
| `v7_5_holdout_evaluation.csv` | Optional |
| `v7_6_holdout_evaluation.csv` | Recommended |
| `v7_6_best_models.csv` or `v7_4_best_models.csv` | Required (skips CV if present) |

## Run Order

```
03 → 05 (Config) → 06 (Panel) → 07 (CV/models) → 08 (Raw forecasts) →
09 (Calibration) → 10 (Grid search) → 11 (Results) → 12 (Top 3) →
13 (Best outputs) → 14 (Validation) → 15 (Comparison) → 16 (Summary)
```

> **Section 10 (Grid search)** runs 10 allocation variants. Each variant
> reallocates in ~seconds — total expected time <5 minutes.